# 🎓 ECG Image Digitization - Training Notebook

This notebook trains the ECG digitization model and saves it to the `output/` folder.

**Output Files:**
- `output/ecg_model.pth` - Trained model weights
- `output/model_config.json` - Model configuration

**Next Step:** Use `ecg_testing.ipynb` for inference with the trained model

In [ ]:
# Environment setup for local execution
import numpy as np
import pandas as pd
import os

# Check if we're running on Kaggle or locally
if os.path.exists('/kaggle/input'):
    print("Running on Kaggle environment")
else:
    print("Running in local environment")
    print("Note: This notebook requires ECG image digitization dataset")
    
    current_dir = os.getcwd()
    print(f"\nCurrent working directory: {current_dir}")
    
    # Create output directory
    os.makedirs('output', exist_ok=True)
    print("✅ Created 'output' folder for saving model")

In [ ]:
# Imports and setup
import pandas as pd
import numpy as np
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from scipy.signal import correlate
import json
import os

# GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# Auto-detect environment and set appropriate paths
if os.path.exists('/kaggle/input'):
    BASE_PATH = Path("../input/physionet-ecg-image-digitization/")
    OUTPUT_PATH = Path("/kaggle/working/output")
    print("🔵 Running on Kaggle")
else:
    possible_paths = [
        Path("./data/physionet-ecg-image-digitization/"),
        Path("./physionet-ecg-image-digitization/"),
        Path("../data/physionet-ecg-image-digitization/"),
        Path("./data/"),
        Path("../data/"),
    ]
    
    BASE_PATH = None
    for path in possible_paths:
        if path.exists():
            BASE_PATH = path
            break
    
    if BASE_PATH is None:
        BASE_PATH = Path("./data/")
        print("🔴 Dataset not found. Creating default data folder.")
        BASE_PATH.mkdir(exist_ok=True)
    else:
        print("🟢 Running locally with dataset found")
    
    OUTPUT_PATH = Path("./output")

OUTPUT_PATH.mkdir(exist_ok=True)
TRAIN_PATH = BASE_PATH / "train"
TEST_PATH = BASE_PATH / "test"

print(f"Base path: {BASE_PATH}")
print(f"Train path: {TRAIN_PATH}")
print(f"Output path: {OUTPUT_PATH}")

In [ ]:
# Load metadata
train_meta = pd.read_csv(BASE_PATH / "train.csv")
print("Train metadata:")
print(train_meta.head())
print(f"\nNumber of training samples: {len(train_meta)}")
print(f"Sampling frequencies: {sorted(train_meta['fs'].unique())}")

In [ ]:
# Utility Functions
import io
from PIL import Image

def load_ecg_csv(record_id):
    """Load full 12-lead ECG time series from train folder"""
    record_id = str(record_id)
    path = TRAIN_PATH / record_id / f"{record_id}.csv"
    
    if not path.exists():
        print(f"⚠️ Warning: CSV file not found at {path}")
        dummy_data = np.random.randn(5000, 12) * 0.5
        return dummy_data
    
    try:
        df = pd.read_csv(path)
        return df.values
    except Exception as e:
        print(f"❌ Error reading CSV: {e}")
        return np.random.randn(5000, 12) * 0.5

def load_ecg_image(record_id, img_number='0001', train=True):
    """Load ECG image as 3-channel RGB"""
    record_id = str(record_id)
    folder = TRAIN_PATH if train else TEST_PATH
    
    if train:
        path = folder / record_id / f"{record_id}-{img_number}.png"
    else:
        path = folder / f"{record_id}.png"
        if not path.exists():
            path = folder / record_id / f"{record_id}.png"
    
    if not path.exists():
        print(f"⚠️ Warning: Image not found at {path}")
        return create_dummy_ecg_image()
    
    try:
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if img is None:
            return create_dummy_ecg_image()
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img
    except Exception as e:
        print(f"❌ Error loading image: {e}")
        return create_dummy_ecg_image()

def create_dummy_ecg_image():
    """Create a dummy ECG-like image for demonstration"""
    try:
        x = np.linspace(0, 10, 800)
        fig, axes = plt.subplots(4, 3, figsize=(12, 8))
        fig.patch.set_facecolor('white')
        
        for i, ax in enumerate(axes.flat):
            signal = np.sin(x * 2 * np.pi) + 0.3 * np.sin(x * 10 * np.pi) + np.random.normal(0, 0.1, len(x))
            ax.plot(x, signal, 'k-', linewidth=1)
            ax.set_xlim(0, 10)
            ax.set_ylim(-2, 2)
            ax.grid(True, alpha=0.3)
            ax.set_title(f'Lead {i+1}', fontsize=8)
            ax.tick_params(labelsize=6)
        
        plt.tight_layout()
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        plt.close()
        buf.seek(0)
        
        pil_img = Image.open(buf)
        img_array = np.array(pil_img)[:, :, :3]
        return img_array
    except Exception as e:
        print(f"❌ Error creating dummy image: {e}")
        return np.ones((600, 800, 3), dtype=np.uint8) * 255

def preprocess_img(img, target_size=(224, 224)):
    """Resize and normalize RGB image for EfficientNet"""
    try:
        resized = cv2.resize(img, target_size)
        normalized = resized.astype(np.float32) / 255.0
        normalized = normalized.transpose(2, 0, 1)
        return normalized
    except Exception as e:
        print(f"❌ Error preprocessing image: {e}")
        return np.random.randn(3, target_size[0], target_size[1]).astype(np.float32)

print("✅ Utility functions loaded successfully!")

In [ ]:
# PyTorch Dataset
class ECGDataset(Dataset):
    def __init__(self, meta_df, train=True, transforms=None, img_number='0001'):
        self.meta = meta_df
        self.train = train
        self.transforms = transforms
        self.img_number = img_number
        
    def __len__(self):
        return len(self.meta)
    
    def __getitem__(self, idx):
        record_id = self.meta.iloc[idx]['id']
        
        img = load_ecg_image(record_id, self.img_number, train=self.train)
        
        if self.transforms and self.train:
            augmented = self.transforms(image=img)
            img = augmented['image']
        
        img = preprocess_img(img)
        img = torch.tensor(img, dtype=torch.float32)
        
        if self.train:
            sig = load_ecg_csv(record_id).T
            sig = torch.tensor(sig, dtype=torch.float32)
            return img, sig, record_id
        else:
            return img, record_id

print("✅ Dataset class loaded!")

In [ ]:
# Data Augmentation
def add_gaussian_noise_transform(img, **kwargs):
    noise = np.random.normal(0.0, 0.01, img.shape).astype(np.float32)
    noisy_img = np.clip(img + noise, 0.0, 1.0)
    return noisy_img

train_transforms = A.Compose([
    A.Affine(translate_percent={"x":0.05,"y":0.05}, scale=(0.95,1.05), rotate=(-3,3), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
    A.GaussianBlur(blur_limit=(3,3), p=0.3),
    A.Lambda(image=add_gaussian_noise_transform, p=0.3)
])

print("✅ Data augmentation configured!")

In [ ]:
# CNN Model: EfficientNet Backbone → 12 leads × variable length
class ECGNet(nn.Module):
    def __init__(self, max_seq_len=5000):
        super().__init__()
        self.max_seq_len = max_seq_len
        
        self.backbone = models.efficientnet_b0(weights='IMAGENET1K_V1')
        self.backbone.classifier = nn.Identity()
        
        self.fc = nn.Linear(1280, 12 * max_seq_len)
        
    def forward(self, x):
        features = self.backbone(x)
        out = self.fc(features)
        out = out.view(-1, 12, self.max_seq_len)
        return out

print("✅ Model architecture defined!")

In [ ]:
# Training Setup
import torch.nn.functional as F

def pad_or_truncate_signal(sig, target_len):
    """Pad or truncate signal to target_len along time dimension."""
    seq_len = sig.shape[1]
    if seq_len < target_len:
        pad_size = target_len - seq_len
        sig = F.pad(sig, (0, pad_size))
    elif seq_len > target_len:
        sig = sig[:, :target_len]
    return sig

def collate_fn_train(batch):
    imgs, sigs, record_ids = zip(*batch)
    imgs = torch.stack(imgs, 0)
    sigs_padded = torch.stack([pad_or_truncate_signal(s, MAX_SEQ_LEN) for s in sigs], 0)
    return imgs, sigs_padded, record_ids

# Hyperparameters
EPOCHS = 10
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
NUM_WORKERS = 0
MAX_SEQ_LEN = 5000

print(f"✅ Training configuration:")
print(f"   • Epochs: {EPOCHS}")
print(f"   • Batch size: {BATCH_SIZE}")
print(f"   • Learning rate: {LEARNING_RATE}")
print(f"   • Max sequence length: {MAX_SEQ_LEN}")

In [ ]:
# Create dataset and dataloader
try:
    train_dataset = ECGDataset(train_meta, train=True, transforms=train_transforms)
    test_sample = train_dataset[0]
    print(f"✅ Successfully loaded training data. Sample shapes: {test_sample[0].shape}, {test_sample[1].shape}")
    
except Exception as e:
    print(f"❌ Error loading training data: {e}")
    print("🔧 Using dummy dataset for demonstration")
    
    class DummyDataset(Dataset):
        def __init__(self, size=10):
            self.size = size
        
        def __len__(self):
            return self.size
        
        def __getitem__(self, idx):
            dummy_img = torch.randn(3, 224, 224)
            dummy_sig = torch.randn(12, MAX_SEQ_LEN)
            return dummy_img, dummy_sig, f"dummy_{idx:03d}"
    
    train_dataset = DummyDataset()

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=NUM_WORKERS, 
    pin_memory=True,
    collate_fn=collate_fn_train if hasattr(train_dataset, 'meta') else None
)

print(f"📊 Training samples: {len(train_dataset)}")
print(f"📦 Batches per epoch: {len(train_loader)}")

In [ ]:
# Initialize model, optimizer, and loss
model = ECGNet(max_seq_len=MAX_SEQ_LEN).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {device}")

In [ ]:
# Training Loop
print(f"\n🚀 Starting training for {EPOCHS} epochs...\n")

training_history = {
    'epochs': [],
    'losses': []
}

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch_data in pbar:
        if len(batch_data) == 3:
            imgs, sigs, record_ids = batch_data
        else:
            imgs, sigs, record_ids = batch_data[0], batch_data[1], [f"dummy_{i}" for i in range(len(batch_data[0]))]
        
        imgs = imgs.to(device)
        sigs = sigs.to(device)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        
        assert outputs.shape == sigs.shape, f"Shape mismatch: {outputs.shape} vs {sigs.shape}"
        
        loss = criterion(outputs, sigs)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.6f}'})
    
    avg_loss = running_loss / len(train_loader)
    training_history['epochs'].append(epoch + 1)
    training_history['losses'].append(avg_loss)
    
    print(f"Epoch {epoch+1}/{EPOCHS}, Avg Loss: {avg_loss:.6f}")

print("\n✅ Training completed!")

In [ ]:
# Save model and configuration
model_path = OUTPUT_PATH / 'ecg_model.pth'
config_path = OUTPUT_PATH / 'model_config.json'

# Save model weights
torch.save(model.state_dict(), model_path)
print(f"✅ Model saved to {model_path}")

# Save configuration
config = {
    'max_seq_len': MAX_SEQ_LEN,
    'model_type': 'EfficientNet-B0',
    'input_size': [224, 224],
    'num_leads': 12,
    'epochs_trained': EPOCHS,
    'final_loss': training_history['losses'][-1] if training_history['losses'] else 0,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE
}

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"✅ Configuration saved to {config_path}")

# Save training history
history_df = pd.DataFrame(training_history)
history_path = OUTPUT_PATH / 'training_history.csv'
history_df.to_csv(history_path, index=False)
print(f"✅ Training history saved to {history_path}")

print("\n" + "="*60)
print("📦 OUTPUT FOLDER CONTENTS:")
print("="*60)
for file in OUTPUT_PATH.iterdir():
    size_mb = file.stat().st_size / (1024 * 1024)
    print(f"   • {file.name} ({size_mb:.2f} MB)")
print("="*60)
print("\n🎯 Next step: Use ecg_testing.ipynb for inference!")

In [ ]:
# Plot training history
if len(training_history['losses']) > 0:
    plt.figure(figsize=(10, 5))
    plt.plot(training_history['epochs'], training_history['losses'], marker='o')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss Over Epochs')
    plt.grid(True, alpha=0.3)
    plt.savefig(OUTPUT_PATH / 'training_loss.png', dpi=100, bbox_inches='tight')
    plt.show()
    print("✅ Training plot saved to output/training_loss.png")
else:
    print("⚠️ No training history to plot")